# 01 - Bronze Ingestion
## Raw CSV → Delta Table

**Purpose:** Load the Kaggle Superstore CSV as-is into a Bronze Delta Table.  
No transformations. No cleaning. Exact copy of source data.

**Input :** Superstore CSV (FileStore or Volume)  
**Output:** `sales_bronze.raw_superstore` Delta Table  
**Layer  :** Bronze (raw)


## 1. Load config

In [0]:
%run ./00_setup_and_config

In [0]:
log("INFO", "01_bronze", "Starting Bronze ingestion")


## 2. Read the CSV with schema inference

In [0]:
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv(SOURCE_PATH)
)

log("INFO", "01_bronze", f"CSV loaded from {SOURCE_PATH}")
print(f"  Rows    : {df_raw.count():,}")
print(f"  Columns : {len(df_raw.columns)}")


## 3. Examine what Spark inferred about the data types

In [0]:
print("=== RAW SCHEMA (Spark-inferred) ===\n")
df_raw.printSchema()


## 4. Visual inspection of raw data

In [0]:
print("=== FIRST 5 ROWS (raw) ===\n")
df_raw.show(5, truncate=False)


## 5. Audit column names for SQL compatibility

In [0]:
print("=== RAW COLUMN NAMES ===\n")
for i, col in enumerate(df_raw.columns):
    has_space   = " " in col
    has_hyphen  = "-" in col
    flag = "⚠️  has space" if has_space else ("⚠️  has hyphen" if has_hyphen else "✅")
    print(f"  [{i:02d}] {col:<30} {flag}")

problem_cols = [c for c in df_raw.columns if " " in c or "-" in c]
print(f"\n  Total columns    : {len(df_raw.columns)}")
print(f"  Need renaming    : {len(problem_cols)}")
print("\n  ℹ️  These will be renamed to snake_case in Silver layer")


## 6. Null and blank audit on raw data — type-aware version

In [0]:
# Problem: comparing col == "" fails on non-string columns (Date, Integer)
#          because Spark tries to cast "" to that column's type.
# Fix   : apply blank-string check only to StringType columns.
#         For all other types, check isNull() only.
#
# This is a common production issue — always profile nulls in a type-safe way.

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

print("=== NULL / BLANK COUNT PER COLUMN (raw) ===\n")

string_cols = {
    f.name for f in df_raw.schema.fields
    if isinstance(f.dataType, StringType)
}

null_exprs = [
    F.count(
        F.when(
            F.col(c).isNull() | (F.col(c) == "") if c in string_cols
            else F.col(c).isNull(),
            c
        )
    ).alias(c)
    for c in df_raw.columns
]

null_counts = df_raw.select(null_exprs).collect()[0]

total_nulls = 0
for col_name, null_count in zip(df_raw.columns, null_counts):
    col_type   = df_raw.schema[col_name].dataType.simpleString()
    flag       = f"  ⚠️  {null_count} nulls/blanks" if null_count > 0 else "  ✅"
    bar        = "█" * min(null_count, 20)
    total_nulls += null_count
    print(f"  {col_name:<30} ({col_type:<10}){flag}  {bar}")

print(f"\n  Total null/blank cells : {total_nulls}")
print(f"  Clean columns          : {sum(1 for n in null_counts if n == 0)}/{len(df_raw.columns)}")
print("\n  ℹ️  Nulls will be handled in Silver layer — Bronze preserves raw state")


## 7. Add pipeline metadata columns before writing to Delta and rename all columns to snake_case before writing to Delta

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

df_bronze = df_raw.withColumns({
    "_source_file"    : F.lit(SOURCE_PATH),
    "_ingested_at"    : F.current_timestamp(),
    "_pipeline_name"  : F.lit(PIPELINE_NAME),
})

log("INFO", "01_bronze", "Metadata columns added: _source_file, _ingested_at, _pipeline_name")
print(f"  Total columns after metadata: {len(df_bronze.columns)}")

In [0]:
column_rename_map = {
    "Row ID"        : "row_id",
    "Order ID"      : "order_id",
    "Order Date"    : "order_date",
    "Ship Date"     : "ship_date",
    "Ship Mode"     : "ship_mode",
    "Customer ID"   : "customer_id",
    "Customer Name" : "customer_name",
    "Segment"       : "segment",
    "Country"       : "country",
    "City"          : "city",
    "State"         : "state",
    "Postal Code"   : "postal_code",
    "Region"        : "region",
    "Product ID"    : "product_id",
    "Category"      : "category",
    "Sub-Category"  : "sub_category",
    "Product Name"  : "product_name",
    "Sales"         : "sales",
    "Quantity"      : "quantity",
    "Discount"      : "discount",
    "Profit"        : "profit",
}

# Apply renames — metadata columns pass through unchanged
df_bronze = df_bronze.toDF(*[
    column_rename_map.get(c, c)
    for c in df_bronze.columns
])

# Verify all columns are now Delta-safe
print("=== COLUMN NAMES AFTER RENAME ===\n")
for i, col in enumerate(df_bronze.columns):
    has_space  = " " in col
    has_hyphen = "-" in col
    flag = "❌ STILL INVALID" if (has_space or has_hyphen) else "✅"
    print(f"  [{i:02d}] {col:<30} {flag}")

log("INFO", "01_bronze", f"Columns renamed to snake_case — {len(column_rename_map)} columns renamed")


## 8. Write the Bronze Delta Table

In [0]:
(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

log("INFO", "01_bronze", f"Bronze Delta Table written: {BRONZE_TABLE}")


## 9. Verify the Delta Table was created correctly

In [0]:
df_verify = spark.table(BRONZE_TABLE)

source_count = df_raw.count()
bronze_count = df_verify.count()
match        = "✅ MATCH" if source_count == bronze_count else "❌ MISMATCH"

print(f"  Source row count : {source_count:,}")
print(f"  Bronze row count : {bronze_count:,}")
print(f"  Verification     : {match}")

if source_count != bronze_count:
    log("ERROR", "01_bronze", f"Row count mismatch! Source={source_count}, Bronze={bronze_count}")
    raise Exception("Bronze write verification failed — row counts do not match")

In [0]:
%sql
SELECT
    order_id,
    order_date,
    customer_name,
    category,
    ROUND(sales, 2)   AS sales,
    ROUND(profit, 2)  AS profit,
    _ingested_at,
    _source_file
FROM sales_bronze.raw_superstore
LIMIT 10

In [0]:
%sql
DESCRIBE HISTORY sales_bronze.raw_superstore

In [0]:
%sql
--Also inspect table details — storage location, format, partition info
DESCRIBE DETAIL sales_bronze.raw_superstore


## 10. Final status log

In [0]:
print("=" * 55)
print("  BRONZE INGESTION — COMPLETE")
print("=" * 55)
print(f"  Source       : {SOURCE_PATH}")
print(f"  Table        : {BRONZE_TABLE}")
print(f"  Rows written : {bronze_count:,}")
print(f"  Columns      : {len(df_bronze.columns)} (incl. 3 metadata)")
print(f"  Format       : Delta")
print(f"  Write mode   : overwrite")
print("=" * 55)

log("INFO", "01_bronze", "Bronze ingestion complete ✅")